# 03 — Model Training (Phase System)

**OandaFX** — Full training pipeline with **phase-based checkpointing** to Google Drive.

**Run on Google Colab with GPU runtime.**

## Phase System
Each phase checkpoints to Drive. If Colab disconnects, just re-run all cells — completed phases skip automatically, and in-progress phases resume from the last checkpoint.

| Phase | Description | Checkpoint |
|-------|-------------|------------|
| 0 | Setup & GPU verification | training_state.json |
| 1 | Data loading | Processed features on Drive |
| 2 | Label generation & split | Split indices on Drive |
| 3 | Transformer training | Per-epoch .pt checkpoints |
| 4 | PPO RL training | Per-250K-step checkpoints |
| 5 | SL/TP scaler fitting | scaler.pkl on Drive |
| 6 | Validation backtest | Metrics JSON |
| 7 | Export & versioning | Final model artifacts |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 0: SETUP & GPU VERIFICATION
# ═══════════════════════════════════════════════════════════════════
import sys, os, json, time, shutil
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = Path('/content/drive/MyDrive/oandafx')

    # Clone the repository
    if not Path('/content/wavebot-v1').exists():
        !git clone https://github.com/Unseengap/wavebot-v1.git /content/wavebot-v1
    sys.path.insert(0, '/content/wavebot-v1')

    # Install deps — pin pandas & numpy to versions compatible with Colab
    !pip install -q "pandas==2.2.2" "numpy<2.2.0" torch stable-baselines3 gymnasium pandas_ta scikit-learn tensorboard tqdm
else:
    DRIVE_BASE = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
    sys.path.insert(0, str(DRIVE_BASE))

# Directory structure on Drive (data saved by notebooks 01 & 02)
PROCESSED_DIR = DRIVE_BASE / 'data' / 'processed'
CHECKPOINT_DIR = DRIVE_BASE / 'checkpoints'
MODELS_DIR = DRIVE_BASE / 'models'
LOGS_DIR = DRIVE_BASE / 'logs'

for d in [PROCESSED_DIR, CHECKPOINT_DIR, MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Verify processed data exists (from notebook 02)
if not (PROCESSED_DIR / 'all_pairs_features.parquet').exists():
    raw_dir = DRIVE_BASE / 'data' / 'raw'
    if not raw_dir.exists() or not list(raw_dir.iterdir()):
        print('❌ No data found on Drive!')
        print('   Run notebook 01 first (download), then notebook 02 (features).')
    else:
        print('⚠️  Processed features not found — raw data exists on Drive.')
        print('   Notebook will attempt feature engineering in Phase 1.')
else:
    print('✅ Processed features found on Drive')

STATE_FILE = CHECKPOINT_DIR / 'training_state.json'

# ─── Training state management ───────────────────────────────────
def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, 'r') as f:
            return json.load(f)
    return {'phases_complete': [], 'current_phase': 0, 'details': {}}

def save_state(state):
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2, default=str)

def phase_complete(phase_num):
    state = load_state()
    return phase_num in state.get('phases_complete', [])

def mark_phase_complete(phase_num, details=None):
    state = load_state()
    if phase_num not in state['phases_complete']:
        state['phases_complete'].append(phase_num)
    state['current_phase'] = phase_num + 1
    if details:
        state['details'][str(phase_num)] = details
    state['last_updated'] = datetime.now(timezone.utc).isoformat()
    save_state(state)

# ─── GPU check & optimization config ────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Auto-detect GPU and set optimal batch size
USE_AMP = False      # Mixed precision (FP16)
BATCH_SIZE = 256     # Default
NUM_WORKERS = 2      # DataLoader workers
COMPILE_MODEL = False

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name} ({vram:.1f} GB VRAM)')

    # Enable mixed precision — ~1.5-2× faster on T4/L4/A100
    USE_AMP = True
    print(f'✅ Mixed precision (FP16): ENABLED')

    # Auto-scale batch size based on VRAM
    if vram >= 35:       # A100 40GB
        BATCH_SIZE = 1024
        NUM_WORKERS = 4
    elif vram >= 20:     # L4 24GB
        BATCH_SIZE = 768
        NUM_WORKERS = 4
    elif vram >= 14:     # T4 15GB
        BATCH_SIZE = 512
        NUM_WORKERS = 4
    else:                # Smaller GPUs
        BATCH_SIZE = 256
        NUM_WORKERS = 2
    print(f'✅ Batch size: {BATCH_SIZE} (auto-scaled for {vram:.0f}GB VRAM)')
    print(f'✅ DataLoader workers: {NUM_WORKERS}')

    # Enable torch.compile for PyTorch 2.0+ (~10-30% faster)
    if hasattr(torch, 'compile'):
        COMPILE_MODEL = True
        print(f'✅ torch.compile: ENABLED (PyTorch {torch.__version__})')

    # Enable cuDNN auto-tuner — finds fastest convolution algorithms
    torch.backends.cudnn.benchmark = True
    print(f'✅ cuDNN benchmark: ENABLED')

    # Enable TF32 on Ampere/Ada GPUs — faster matmul with minimal precision loss
    if 'A100' in gpu_name or 'A10' in gpu_name or 'L4' in gpu_name or 'RTX 30' in gpu_name or 'RTX 40' in gpu_name:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print(f'✅ TF32 matmul: ENABLED (Ampere/Ada GPU detected)')
else:
    gpu_name = 'cpu'
    vram = 0
    print('⚠️  No GPU detected — training will be slow. Enable GPU runtime in Colab.')

print(f'\nDevice: {device}')
print(f'Drive base: {DRIVE_BASE}')
print(f'Training state: {load_state()}')

mark_phase_complete(0, {
    'gpu': str(device),
    'gpu_name': gpu_name,
    'vram_gb': round(vram, 1) if torch.cuda.is_available() else 0,
    'batch_size': BATCH_SIZE,
    'amp_enabled': USE_AMP,
    'compile_enabled': COMPILE_MODEL,
    'start_time': datetime.now(timezone.utc).isoformat(),
})

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 1: DATA LOADING (from Google Drive)
# ═══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np

if phase_complete(1):
    print('Phase 1 already complete — loading cached data from Drive...')
    combined_path = PROCESSED_DIR / 'all_pairs_features.parquet'
    all_features = pd.read_parquet(combined_path)

    with open(PROCESSED_DIR / 'feature_columns.json', 'r') as f:
        feature_cols = json.load(f)

    pairs = all_features['pair'].unique().tolist() if 'pair' in all_features.columns else ['unknown']
    print(f'Loaded: {len(all_features):,} rows × {all_features.shape[1]} cols, {len(feature_cols)} features')
else:
    print('Phase 1: Loading data from Google Drive...')

    # Try loading pre-processed features (saved by notebook 02)
    combined_path = PROCESSED_DIR / 'all_pairs_features.parquet'
    cols_path = PROCESSED_DIR / 'feature_columns.json'

    if combined_path.exists() and cols_path.exists():
        print('✅ Found processed features on Drive (from notebook 02) — loading...')
        all_features = pd.read_parquet(combined_path)
        with open(cols_path, 'r') as f:
            feature_cols = json.load(f)
    else:
        # Fallback: run feature engineering from raw data on Drive
        print('⚠️  No processed features found — running feature engineering from raw data...')
        RAW_DIR = DRIVE_BASE / 'data' / 'raw'

        if not RAW_DIR.exists() or not list(RAW_DIR.iterdir()):
            raise FileNotFoundError(
                'No data on Drive! Run notebook 01 (download) and notebook 02 (features) first.'
            )

        from src.data.storage import load_candles
        from src.features.engineer import FeatureEngineer

        pairs = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])

        engineer = FeatureEngineer(
            base_timeframe='M15',
            higher_timeframes=['H1', 'H4', 'D'],
            lookback_bars=128,
            include_patterns=True,
            include_mtf=True,
        )

        feature_datasets = {}
        for pair in pairs:
            print(f'  Processing {pair}...')
            raw_data = {}
            for tf in ['M15', 'H1', 'H4', 'D']:
                path = RAW_DIR / pair / f'{tf}.parquet'
                if path.exists():
                    raw_data[tf] = load_candles(path)

            if 'M15' in raw_data:
                df = engineer.build(raw_data)
                df['pair'] = pair
                feature_datasets[pair] = df
                print(f'    {df.shape[0]:,} rows × {df.shape[1]} features')

        all_features = pd.concat(feature_datasets.values(), ignore_index=True)
        feature_cols = engineer.get_feature_columns(all_features)

        # Save to Drive for future runs
        all_features.to_parquet(combined_path, index=False)
        with open(cols_path, 'w') as f:
            json.dump(feature_cols, f)
        print(f'Saved processed features to Drive')

    pairs = all_features['pair'].unique().tolist() if 'pair' in all_features.columns else ['unknown']
    print(f'\n✅ Phase 1 complete: {len(all_features):,} rows, {len(feature_cols)} features, {len(pairs)} pairs')
    mark_phase_complete(1, {'rows': len(all_features), 'features': len(feature_cols), 'pairs': pairs})

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 2: LABEL GENERATION & TRAIN/VAL/TEST SPLIT
# ═══════════════════════════════════════════════════════════════════
from src.models.transformer import generate_labels

SPLIT_FILE = CHECKPOINT_DIR / 'split_indices.json'

if phase_complete(2):
    print('Phase 2 already complete — loading split indices...')
    with open(SPLIT_FILE, 'r') as f:
        split_info = json.load(f)
    train_end = split_info['train_end']
    val_end = split_info['val_end']
else:
    print('Phase 2: Generating labels and splitting data...')
    
    # Generate labels if not already present
    if 'label_h5' not in all_features.columns:
        print('  Generating directional labels (horizons 1, 3, 5)...')
        all_features = generate_labels(all_features, horizons=[1, 3, 5], threshold_atr_multiplier=0.5)
    
    # Check label distribution
    for h in [1, 3, 5]:
        col = f'label_h{h}'
        if col in all_features.columns:
            dist = all_features[col].value_counts(normalize=True)
            print(f'  Horizon {h}: Long={dist.get("long", 0):.1%}  Short={dist.get("short", 0):.1%}  Flat={dist.get("flat", 0):.1%}')
    
    # Time-ordered split (NO shuffling — prevents look-ahead bias)
    all_features = all_features.sort_values('time').reset_index(drop=True)
    n = len(all_features)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    
    split_info = {'train_end': train_end, 'val_end': val_end, 'total': n}
    with open(SPLIT_FILE, 'w') as f:
        json.dump(split_info, f)
    
    # Save labeled data back to Drive
    all_features.to_parquet(PROCESSED_DIR / 'all_pairs_features.parquet', index=False)
    
    print(f'\n✅ Phase 2 complete')
    mark_phase_complete(2, split_info)

# Create splits
train_df = all_features.iloc[:train_end]
val_df = all_features.iloc[train_end:val_end]
test_df = all_features.iloc[val_end:]

print(f'Train: {len(train_df):>10,} rows | {train_df["time"].min()} → {train_df["time"].max()}')
print(f'Val:   {len(val_df):>10,} rows | {val_df["time"].min()} → {val_df["time"].max()}')
print(f'Test:  {len(test_df):>10,} rows | {test_df["time"].min()} → {test_df["time"].max()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 3: TRANSFORMER TRAINING (GPU-optimized)
# ═══════════════════════════════════════════════════════════════════
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from src.models.transformer import ForexTransformer, ForexSequenceDataset

TRANSFORMER_DIR = CHECKPOINT_DIR / 'transformer'
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)
BEST_TRANSFORMER_PATH = TRANSFORMER_DIR / 'best_transformer.pt'

# Hyperparameters (BATCH_SIZE and NUM_WORKERS set in Phase 0 based on GPU)
LOOKBACK = 128
MAX_EPOCHS = 100
PATIENCE = 10
LR = 1e-4

if phase_complete(3):
    print('Phase 3 already complete — loading best Transformer from Drive...')
    model = ForexTransformer(
        input_dim=len(feature_cols), d_model=256, nhead=8,
        num_layers=4, dim_feedforward=1024, dropout=0.1,
        num_classes=3, num_horizons=3,
    ).to(device)
    model.load_state_dict(torch.load(BEST_TRANSFORMER_PATH, map_location=device))
    print(f'Loaded Transformer: {sum(p.numel() for p in model.parameters()):,} params')
else:
    print('Phase 3: Training Transformer (GPU-optimized)...')
    phase3_start = time.time()

    # ── Create datasets with optimized DataLoaders ───────────────
    train_dataset = ForexSequenceDataset(train_df, feature_cols, 'label_h5', lookback=LOOKBACK)
    val_dataset = ForexSequenceDataset(val_df, feature_cols, 'label_h5', lookback=LOOKBACK)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,      # Keep workers alive between epochs (faster)
        prefetch_factor=3,             # Pre-load 3 batches per worker
        drop_last=True,                # Even batch sizes for GPU efficiency
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE * 2,     # Double batch for val (no gradients = more VRAM)
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
    )

    print(f'Train: {len(train_loader)} batches × {BATCH_SIZE} = {len(train_loader)*BATCH_SIZE:,} samples/epoch')
    print(f'Val:   {len(val_loader)} batches × {BATCH_SIZE*2}')

    # ── Initialize model ─────────────────────────────────────────
    model = ForexTransformer(
        input_dim=len(feature_cols), d_model=256, nhead=8,
        num_layers=4, dim_feedforward=1024, dropout=0.1,
        num_classes=3, num_horizons=3,
    ).to(device)

    # Compile model for faster execution (PyTorch 2.0+)
    if COMPILE_MODEL:
        model = torch.compile(model, mode='reduce-overhead')
        print(f'✅ Model compiled with torch.compile')

    param_count = sum(p.numel() for p in model.parameters())
    print(f'Model parameters: {param_count:,}')

    # ── Optimizer with fused kernels ─────────────────────────────
    # fused=True runs optimizer step on GPU (faster than CPU)
    try:
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4, fused=True)
        print(f'✅ Fused AdamW optimizer (GPU-accelerated)')
    except TypeError:
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    # OneCycleLR — faster convergence than CosineAnnealing
    scheduler = OneCycleLR(
        optimizer,
        max_lr=LR * 10,               # Peak at 10× base LR
        epochs=MAX_EPOCHS,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,                 # Warmup for first 30% of training
        anneal_strategy='cos',
        div_factor=10,
        final_div_factor=100,
    )

    # ── Class weights for imbalanced labels ──────────────────────
    from sklearn.utils.class_weight import compute_class_weight
    labels_array = np.array(train_dataset.encoded_labels[LOOKBACK:])
    class_weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=labels_array)
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)

    # ── Mixed precision scaler ───────────────────────────────────
    amp_scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    if USE_AMP:
        print(f'✅ Mixed precision (AMP) enabled — ~1.5-2× speedup')

    # ── Check for existing checkpoint to resume ──────────────────
    start_epoch = 0
    best_val_loss = float('inf')
    patience_counter = 0

    checkpoint_files = sorted(TRANSFORMER_DIR.glob('epoch_*.pt'))
    if checkpoint_files:
        latest_ckpt = checkpoint_files[-1]
        print(f'  Resuming from checkpoint: {latest_ckpt.name}')
        ckpt = torch.load(latest_ckpt, map_location=device)
        # Load into base model (handle compiled model wrapper)
        base_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        base_model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        patience_counter = ckpt.get('patience_counter', 0)
        if 'amp_scaler_state' in ckpt and USE_AMP:
            amp_scaler.load_state_dict(ckpt['amp_scaler_state'])
        # Re-create OneCycleLR at correct step
        scheduler = OneCycleLR(
            optimizer, max_lr=LR * 10, epochs=MAX_EPOCHS,
            steps_per_epoch=len(train_loader), pct_start=0.3,
            anneal_strategy='cos', div_factor=10, final_div_factor=100,
            last_epoch=start_epoch * len(train_loader) - 1,
        )
        print(f'  Resuming from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}')

    # ── Training loop ────────────────────────────────────────────
    print(f'\n{"="*70}')
    print(f'  GPU: {gpu_name if torch.cuda.is_available() else "CPU"} | '
          f'AMP: {USE_AMP} | Batch: {BATCH_SIZE} | '
          f'Workers: {NUM_WORKERS} | Compiled: {COMPILE_MODEL}')
    print(f'{"="*70}\n')

    epoch_times = []

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_start = time.time()

        # ── Train ────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()

            # Mixed precision forward pass
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(batch_x)
                loss = criterion(logits, batch_y)

            # Mixed precision backward pass
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()

            scheduler.step()  # OneCycleLR steps per batch, not per epoch

            train_loss += loss.item() * batch_x.size(0)
            train_correct += (logits.argmax(dim=1) == batch_y).sum().item()
            train_total += batch_x.size(0)

        # ── Validate ─────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                for batch_x, batch_y in val_loader:
                    batch_x = batch_x.to(device, non_blocking=True)
                    batch_y = batch_y.to(device, non_blocking=True)
                    logits = model(batch_x)
                    loss = criterion(logits, batch_y)
                    val_loss += loss.item() * batch_x.size(0)
                    val_correct += (logits.argmax(dim=1) == batch_y).sum().item()
                    val_total += batch_x.size(0)

        train_loss /= max(train_total, 1)
        val_loss /= max(val_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_acc = val_correct / max(val_total, 1)
        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)

        # GPU memory stats
        if torch.cuda.is_available():
            gpu_mem = torch.cuda.max_memory_allocated() / 1e9
            gpu_util = f'{gpu_mem:.1f}GB'
        else:
            gpu_util = 'N/A'

        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:3d}/{MAX_EPOCHS} | '
              f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
              f'Acc: {train_acc:.3f}/{val_acc:.3f} | '
              f'LR: {current_lr:.2e} | '
              f'{epoch_time:.1f}s | GPU: {gpu_util}')

        # ── Checkpoint to Drive ──────────────────────────────────
        base_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        ckpt_data = {
            'epoch': epoch,
            'model_state_dict': base_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'best_val_loss': best_val_loss,
            'patience_counter': patience_counter,
            'amp_scaler_state': amp_scaler.state_dict() if USE_AMP else None,
        }
        torch.save(ckpt_data, TRANSFORMER_DIR / f'epoch_{epoch:03d}.pt')

        # Keep only last 3 checkpoints to save space
        old_ckpts = sorted(TRANSFORMER_DIR.glob('epoch_*.pt'))[:-3]
        for old in old_ckpts:
            old.unlink()

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(base_model.state_dict(), BEST_TRANSFORMER_PATH)
            print(f'  ✓ New best model saved (val_loss={val_loss:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # ── Training summary ─────────────────────────────────────────
    total_time = time.time() - phase3_start
    avg_epoch_time = sum(epoch_times) / len(epoch_times) if epoch_times else 0

    # Load best model (unwrap compiled if needed)
    base_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    base_model.load_state_dict(torch.load(BEST_TRANSFORMER_PATH, map_location=device))
    model = base_model  # Use uncompiled for downstream compatibility

    print(f'\n{"="*70}')
    print(f'✅ Phase 3 complete — Transformer training')
    print(f'   Best val_loss: {best_val_loss:.4f}')
    print(f'   Epochs trained: {epoch + 1}')
    print(f'   Avg epoch time: {avg_epoch_time:.1f}s')
    print(f'   Total time: {total_time/60:.1f} minutes')
    if torch.cuda.is_available():
        print(f'   Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
    print(f'{"="*70}')

    # Clear GPU cache before PPO
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    mark_phase_complete(3, {
        'best_val_loss': float(best_val_loss),
        'epochs_trained': epoch + 1,
        'total_time_minutes': round(total_time / 60, 1),
        'avg_epoch_seconds': round(avg_epoch_time, 1),
        'batch_size': BATCH_SIZE,
        'amp_enabled': USE_AMP,
    })

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 4: PPO REINFORCEMENT LEARNING TRAINING (GPU-optimized)
# ═══════════════════════════════════════════════════════════════════
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from src.models.rl_agent import OandaForexEnv

PPO_DIR = CHECKPOINT_DIR / 'ppo'
PPO_DIR.mkdir(parents=True, exist_ok=True)
BEST_PPO_PATH = PPO_DIR / 'best_ppo'

TOTAL_TIMESTEPS = 5_000_000
CHECKPOINT_FREQ = 250_000

# Auto-scale PPO based on GPU
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 35:       # A100
        N_ENVS = 8
        PPO_BATCH = 256
        PPO_N_STEPS = 4096
    elif vram >= 20:     # L4 24GB
        N_ENVS = 8
        PPO_BATCH = 192
        PPO_N_STEPS = 3072
    elif vram >= 14:     # T4
        N_ENVS = 8
        PPO_BATCH = 128
        PPO_N_STEPS = 2048
    else:
        N_ENVS = 4
        PPO_BATCH = 64
        PPO_N_STEPS = 2048
else:
    N_ENVS = 4
    PPO_BATCH = 64
    PPO_N_STEPS = 2048

if phase_complete(4):
    print('Phase 4 already complete — loading best PPO agent from Drive...')
    ppo_agent = PPO.load(str(BEST_PPO_PATH), device=device)
    print('PPO agent loaded.')
else:
    print('Phase 4: Training PPO RL agent (GPU-optimized)...')
    phase4_start = time.time()

    # Ensure transformer is in eval mode
    model.eval()

    # Create environments
    def make_env(data, seed):
        def _init():
            env = OandaForexEnv(
                features_df=data,
                feature_cols=feature_cols,
                transformer=model,
                spread_pips=1.2,
                pip_value=10.0,
                max_leverage=50,
                initial_balance=10_000,
                lookback=LOOKBACK,
            )
            env.reset(seed=seed)
            return env
        return _init

    # SubprocVecEnv runs each env in a separate process — true parallelism
    # Falls back to DummyVecEnv if subprocess spawning fails (some Colab envs)
    try:
        train_envs = SubprocVecEnv([make_env(train_df, seed=i) for i in range(N_ENVS)])
        env_type = 'SubprocVecEnv'
    except Exception as e:
        print(f'  SubprocVecEnv failed ({e}), falling back to DummyVecEnv')
        train_envs = DummyVecEnv([make_env(train_df, seed=i) for i in range(N_ENVS)])
        env_type = 'DummyVecEnv'

    eval_env = OandaForexEnv(
        features_df=val_df,
        feature_cols=feature_cols,
        transformer=model,
        spread_pips=1.2,
        pip_value=10.0,
        max_leverage=50,
        initial_balance=10_000,
        lookback=LOOKBACK,
    )

    print(f'  Envs: {N_ENVS} × {env_type}')
    print(f'  PPO batch: {PPO_BATCH} | n_steps: {PPO_N_STEPS}')
    print(f'  Effective batch per update: {N_ENVS * PPO_N_STEPS:,} timesteps')

    # ── Check for existing PPO checkpoint to resume ──────────────
    ppo_checkpoints = sorted(PPO_DIR.glob('ppo_oandafx_*_steps.zip'))

    if ppo_checkpoints:
        latest_ppo = str(ppo_checkpoints[-1]).replace('.zip', '')
        step_str = ppo_checkpoints[-1].stem.split('_')[-2]
        completed_steps = int(step_str)
        remaining_steps = max(TOTAL_TIMESTEPS - completed_steps, 0)

        print(f'  Resuming from checkpoint: {ppo_checkpoints[-1].name} ({completed_steps:,} steps done)')
        print(f'  Remaining: {remaining_steps:,} steps')

        ppo_agent = PPO.load(latest_ppo, env=train_envs, device=device)
    else:
        remaining_steps = TOTAL_TIMESTEPS
        ppo_agent = PPO(
            policy='MlpPolicy',
            env=train_envs,
            learning_rate=3e-4,
            n_steps=PPO_N_STEPS,
            batch_size=PPO_BATCH,
            n_epochs=10,
            gamma=0.99,
            gae_lambda=0.95,
            clip_range=0.2,
            ent_coef=0.01,
            vf_coef=0.5,
            max_grad_norm=0.5,
            verbose=1,
            tensorboard_log=str(LOGS_DIR / 'ppo'),
            device=device,
            policy_kwargs=dict(
                net_arch=dict(pi=[256, 256, 128], vf=[256, 256, 128]),
            ),
        )

    if remaining_steps > 0:
        # Callbacks
        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path=str(BEST_PPO_PATH.parent),
            log_path=str(PPO_DIR / 'eval_logs'),
            eval_freq=max(50_000 // N_ENVS, 1),  # Adjust for n_envs
            n_eval_episodes=5,
            deterministic=True,
        )
        checkpoint_callback = CheckpointCallback(
            save_freq=max(CHECKPOINT_FREQ // N_ENVS, 1),
            save_path=str(PPO_DIR),
            name_prefix='ppo_oandafx',
        )

        print(f'\n  Training PPO for {remaining_steps:,} timesteps...')
        ppo_agent.learn(
            total_timesteps=remaining_steps,
            callback=[eval_callback, checkpoint_callback],
            progress_bar=True,
        )

    # Save final model
    ppo_agent.save(str(BEST_PPO_PATH))
    train_envs.close()

    total_time = time.time() - phase4_start
    print(f'\n{"="*70}')
    print(f'✅ Phase 4 complete — PPO RL training')
    print(f'   Total timesteps: {TOTAL_TIMESTEPS:,}')
    print(f'   Parallel envs: {N_ENVS} ({env_type})')
    print(f'   Total time: {total_time/60:.1f} minutes')
    print(f'{"="*70}')

    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    mark_phase_complete(4, {
        'total_timesteps': TOTAL_TIMESTEPS,
        'n_envs': N_ENVS,
        'env_type': env_type,
        'ppo_batch': PPO_BATCH,
        'ppo_n_steps': PPO_N_STEPS,
        'total_time_minutes': round(total_time / 60, 1),
    })

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 5: SL/TP SCALER FITTING
# ═══════════════════════════════════════════════════════════════════
import pickle
from sklearn.preprocessing import StandardScaler

SCALER_PATH = CHECKPOINT_DIR / 'sl_tp_scaler.pkl'

if phase_complete(5):
    print('Phase 5 already complete — loading scaler from Drive...')
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
else:
    print('Phase 5: Fitting SL/TP scaler...')
    
    sl_tp_features = ['atr_14', 'atr_7_ratio', 'atr_50_ratio', 'vol_regime',
                      'sr_nearest_support_dist', 'sr_nearest_resistance_dist']
    available = [f for f in sl_tp_features if f in train_df.columns]
    
    scaler = StandardScaler()
    scaler.fit(train_df[available].dropna())
    
    with open(SCALER_PATH, 'wb') as f:
        pickle.dump(scaler, f)
    
    print(f'Scaler fitted on {len(available)} features: {available}')
    print(f'\n✅ Phase 5 complete')
    mark_phase_complete(5, {'scaler_features': available})

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 6: VALIDATION BACKTEST
# ═══════════════════════════════════════════════════════════════════
from src.backtest.engine import WalkForwardBacktest
from src.backtest.metrics import compute_metrics
from src.models.sl_tp_module import DynamicSLTP

if phase_complete(6):
    print('Phase 6 already complete — loading validation metrics...')
    metrics_path = CHECKPOINT_DIR / 'validation_metrics.json'
    if metrics_path.exists():
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
else:
    print('Phase 6: Running validation backtest on test set...')
    
    sl_tp_module = DynamicSLTP()
    
    backtest = WalkForwardBacktest(
        features_df=test_df,
        feature_cols=feature_cols,
        transformer=model,
        sl_tp_module=sl_tp_module,
        spread_pips=1.2,
        initial_balance=10_000,
    )
    
    trades = backtest.run()
    metrics = compute_metrics(trades)
    
    print('\n' + '=' * 60)
    print('VALIDATION BACKTEST RESULTS')
    print('=' * 60)
    print(f'  Total return:      {metrics["total_return"]:.2%}')
    print(f'  Annualized return: {metrics["annualized_return"]:.2%}')
    print(f'  Sharpe ratio:      {metrics["sharpe"]:.2f}')
    print(f'  Sortino ratio:     {metrics["sortino"]:.2f}')
    print(f'  Max drawdown:      {metrics["max_drawdown"]:.2%}')
    print(f'  Win rate:          {metrics["win_rate"]:.2%}')
    print(f'  Profit factor:     {metrics["profit_factor"]:.2f}')
    print(f'  Total trades:      {metrics["total_trades"]}')
    print(f'  Avg duration:      {metrics["avg_duration_bars"]:.0f} bars')
    print('=' * 60)
    
    # Deployment threshold check
    THRESHOLDS = {'sharpe': 1.0, 'max_drawdown': -0.15, 'profit_factor': 1.2, 'win_rate': 0.45}
    deploy_ok = (
        metrics['sharpe'] >= THRESHOLDS['sharpe']
        and metrics['max_drawdown'] >= THRESHOLDS['max_drawdown']
        and metrics['profit_factor'] >= THRESHOLDS['profit_factor']
        and metrics['win_rate'] >= THRESHOLDS['win_rate']
    )
    
    if deploy_ok:
        print('\n✅ Model PASSES deployment thresholds — safe to deploy.')
    else:
        print('\n❌ Model FAILS deployment thresholds — review before deploying.')
    
    metrics['deploy_approved'] = deploy_ok
    metrics_path = CHECKPOINT_DIR / 'validation_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print(f'\n✅ Phase 6 complete')
    mark_phase_complete(6, metrics)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 7: EXPORT & VERSIONING
# ═══════════════════════════════════════════════════════════════════
if phase_complete(7):
    print('Phase 7 already complete — model already exported.')
    state = load_state()
    details = state.get('details', {}).get('7', {})
    print(f'Version: {details.get("version", "unknown")}')
    print(f'Path: {details.get("save_dir", "unknown")}')
else:
    print('Phase 7: Exporting model artifacts...')
    
    VERSION = '1.0.0'  # Increment for each new training run
    SAVE_DIR = MODELS_DIR / f'v{VERSION}'
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    
    # Save Transformer
    torch.save(model.state_dict(), SAVE_DIR / 'transformer.pt')
    
    # Save PPO agent
    ppo_agent.save(str(SAVE_DIR / 'ppo_agent'))
    
    # Save SL/TP scaler
    shutil.copy(SCALER_PATH, SAVE_DIR / 'sl_tp_scaler.pkl')
    
    # Save feature columns
    with open(SAVE_DIR / 'feature_columns.json', 'w') as f:
        json.dump(feature_cols, f)
    
    # Save metadata
    metadata = {
        'version': VERSION,
        'training_date': datetime.now(timezone.utc).strftime('%Y-%m-%d'),
        'training_data_range': [
            str(train_df['time'].min().date()),
            str(train_df['time'].max().date()),
        ],
        'pairs_trained_on': pairs,
        'base_timeframe': 'M15',
        'feature_count': len(feature_cols),
        'transformer_params': sum(p.numel() for p in model.parameters()),
        'ppo_timesteps': TOTAL_TIMESTEPS,
        'validation_metrics': metrics,
        'deploy_approved': metrics.get('deploy_approved', False),
        'ensemble_weights': {'transformer': 0.6, 'rl': 0.4},
        'gpu_used': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    }
    with open(SAVE_DIR / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    # Update latest symlink
    latest_link = MODELS_DIR / 'latest'
    if latest_link.exists() or latest_link.is_symlink():
        latest_link.unlink()
    try:
        latest_link.symlink_to(SAVE_DIR)
    except OSError:
        # Symlinks may not work on Drive — copy instead
        if latest_link.exists():
            shutil.rmtree(latest_link)
        shutil.copytree(SAVE_DIR, latest_link)
    
    print(f'\n✅ Phase 7 complete — Model v{VERSION} saved to Drive')
    print(f'   Path: {SAVE_DIR}')
    print(f'   Files: {[f.name for f in SAVE_DIR.iterdir()]}')
    mark_phase_complete(7, {'version': VERSION, 'save_dir': str(SAVE_DIR)})

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  TRAINING COMPLETE — SUMMARY
# ═══════════════════════════════════════════════════════════════════
state = load_state()
print('\n' + '=' * 60)
print('TRAINING PIPELINE COMPLETE')
print('=' * 60)
print(f'Phases completed: {state["phases_complete"]}')
for phase_num, details in state.get('details', {}).items():
    print(f'  Phase {phase_num}: {json.dumps(details, default=str)[:100]}')
print('\nModel artifacts saved to Google Drive.')
print('To deploy: copy model files to VPS or push to GitHub.')
print('=' * 60)